In [81]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
import os
os.chdir(r'C:\Users\Lenovo\Desktop\Diabetes_Classifier\notebooks')
os.chdir("..")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [83]:
from modeling.XGBoost import XGBoost
from modeling.RandomForest import RandomForest
from modeling.KNN import KNN
from modeling.MLP import MLP
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,accuracy_score

In [84]:
PATH="data/preprocessed/final.csv"
df=pd.read_csv(PATH)
df.head()
df.columns = df.columns.str.strip()

In [85]:


y=df['diagnosis']
X=df.drop(['diagnosis'],axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [86]:

xgb=XGBoost()
xgb_params={'n_estimators': 7300, 'max_depth': 9,'learning_rate': 0.042826934350783406, 'gamma': 0.8864325835995647, 'min_child_weight': 1, 'reg_alpha': 0.00019361810915135758, 'reg_lambda': 3.6728295355053825e-06, 'subsample': 0.6459234341228212, 'colsample_bytree': 0.2968183856347904, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)
pd.set_option('display.max_rows',None)

# OOF
# X=xgb.oof(X,y)

#Optuna
# params=xgb.hyperparameter_tuning(X,y,30)
# xgb.set_params(params)

# Train and Test XGB
metrics=xgb.evaluation(X_train,y_train,5)
metrics.out()

xgb.feature_importance(X.columns)

preds=xgb.predict(X_test)
score=roc_auc_score(y_test,preds)
print(score)



Accuracy: 0.8192448233861146
Precision: 0.8122945084796225
F1: 0.807752977552823
Recall: 0.8052776186076726
ROC-AUC: 0.9103467849377775
           Feature  Importance
16  cluster_labels    0.186154
0              age    0.078831
19       Cluster 2    0.078786
10       Age_x_BMI    0.077820
6              ldl    0.037538
13     BMI/HDL+LDL    0.036599
22       Cluster 5    0.036539
2              bmi    0.033637
17       Cluster 0    0.032817
5              hdl    0.032003
21       Cluster 4    0.030113
11       HDL_x_LDL    0.029599
12         BMI/LDL    0.029130
18       Cluster 1    0.027666
7               cr    0.027415
3             chol    0.027043
9           lipids    0.026360
1           gender    0.025974
15        chol/ldl    0.025959
14        bun_x_cr    0.025837
4               tg    0.025189
8              bun    0.025050
20       Cluster 3    0.023514
23   anomaly_score    0.020428
0.8060758646514704


In [87]:
rf=RandomForest()
rf_params={'n_estimators': 600, 'max_depth': 41, 'min_samples_split': 17, 'min_samples_leaf': 6, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'log_loss', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
rf.set_params(rf_params)

metrics=rf.evaluation(X_train,y_train,5)
metrics.out()
# rf_params=rf.hyperparameter_tuning(X_train,y_train,20)
rf.feature_importance(X_train.columns)


Accuracy: 0.8263093788063338
Precision: 0.8198882268579716
F1: 0.8152958605258533
Recall: 0.8125560538356785
ROC-AUC: 0.9161125065931696
           Feature  Importance
10       Age_x_BMI    0.174764
0              age    0.135423
19       Cluster 2    0.120691
22       Cluster 5    0.054190
16  cluster_labels    0.043643
18       Cluster 1    0.040765
17       Cluster 0    0.040168
5              hdl    0.039963
2              bmi    0.036837
11       HDL_x_LDL    0.033430
9           lipids    0.029594
13     BMI/HDL+LDL    0.029591
6              ldl    0.027936
7               cr    0.026618
4               tg    0.026032
21       Cluster 4    0.024602
15        chol/ldl    0.021876
20       Cluster 3    0.019161
12         BMI/LDL    0.018900
3             chol    0.018878
14        bun_x_cr    0.018809
8              bun    0.015034
1           gender    0.002506
23   anomaly_score    0.000588


In [88]:
mlp=MLP()
mlp_params={'hidden_layer_sizes': (128, 64, 32), 'activation': 'relu', 'solver': 'adam', 'alpha': 3.833957124267705e-05, 'learning_rate_init': 0.001020282896183049, 'batch_size': 32, 'max_iter': 500, 'early_stopping': True, 'random_state': 42}
mlp.set_params(mlp_params)

X_train_scaled,y_train_scaled=mlp.scaling(X_train,y_train)
metrics=mlp.evaluation(X_train_scaled,y_train_scaled,5)
metrics.out()

# params=mlp.hyperparameter_tuning(X_train_scaled,y_train_scaled,20)
# mlp.set_params(params)


Accuracy: 0.8243605359317906
Precision: 0.8181108825360859
F1: 0.8137071913776601
Recall: 0.8127963871645469
ROC-AUC: 0.912770024707048


In [89]:
knn=KNN()
knn_params={'n_neighbors': 50, 'weights': 'uniform', 'metric': 'manhattan', 'p': 3, 'algorithm': 'auto', 'leaf_size': 34, 'n_jobs': -1}
knn.set_params(knn_params)

metrics=knn.evaluation(X_train_scaled,y_train_scaled,5)
metrics.out()

# parameters=knn.hyperparameter_tuning(X_train_scaled,y_train_scaled,20)

Accuracy: 0.8138855054811206
Precision: 0.8125853213564602
F1: 0.7976147584949301
Recall: 0.7906617741954994
ROC-AUC: 0.9095757297751735


In [90]:
ensemble=[xgb,rf,mlp,knn]
for model in ensemble:
    if model=='xgb' or model=='rf':
        X_train[f'OOF_{model}']=model.oof(X_train,y_train,model,10)
    else:
        X_train[f'OOF_{model}']=model.oof(X_train_scaled,y_train_scaled,model,10)

<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 1/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 2/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 3/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 4/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 5/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 6/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 7/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 8/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 9/10 done
<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550> fold 10/10 done
<modeling.RandomForest.RandomForest object at 0x000001BC084F51D0> fold 1/10 done
<modeling.RandomForest.RandomForest object at 0x000001BC084F51D0> fold 2/10 done
<modeling.RandomForest.RandomForest object at 0x000001BC084F51D0> fold 3/10 done
<modeling.RandomForest.RandomForest object at 

In [91]:
X_train.shape
# xgb.train(X_train,y_train)
# rf.train(X_train,y_train)
# mlp.train(X_train_scaled,y_train_scaled)
# knn.train(X_train_scaled,y_train_scaled)

(4105, 28)

In [93]:

X_train.head(2)

,age,gender,bmi,chol,tg,hdl,ldl,cr,bun,lipids,...,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,anomaly_score,OOF_<modeling.XGBoost.XGBoost object at 0x000001BC1F9D1550>,OOF_<modeling.RandomForest.RandomForest object at 0x000001BC084F51D0>,OOF_<modeling.MLP.MLP object at 0x000001BC08EE4FC0>,OOF_<modeling.KNN.KNN object at 0x000001BC084F7B10>
3740,51,1,33,3.30,2.00,1.00,1.20,76.0,10.4,0.833333,...,10.963278,8.038825,24.362952,5.367746,8.254704,2,0.584181,0.561582,0.715359,0.5
949,49,1,20,6.54,0.81,2.03,2.74,90.0,6.6,0.740876,...,6.034430,3.324360,24.975450,8.337031,3.095528,0,0.995027,0.989973,0.989578,1.0


In [92]:
# xgb.save_model(xgb,"models/XGBoostClassifier.pkl")